# 第 10 课：FastAPI、流式输出、安全与部署

预计用时：90 分钟  
适合人群：完成上一课的零基础学习者；本 Notebook 也包含独立运行所需的准备代码。

## 学习目标

- 把 Agent 暴露为 HTTP API
- 理解 SSE 流式响应
- 加入输入校验、超时、基础防护和成本路由

## 学习方式

按顺序运行每个代码单元格。先阅读预测结果，再运行验证；遇到报错先看本课“常见问题”，不要直接跳过。带有真实模型或外网请求的示例默认注释，确认 API Key 与费用后再启用。


## 1. 先理解概念

部署意味着 Agent 开始接收不可信输入并消耗真实资源。API 层必须限制请求体、鉴权、限流、超时和日志内容。关键词检测只能做演示，不能代替权限隔离与工具白名单。

### 本课路线

1. 定义请求模型与健康检查
2. 实现普通 chat 接口
3. 实现 SSE 输出
4. 拒绝明显恶意输入
5. 了解模型路由与容器部署清单


## 2. 运行前检查

1. 从项目根目录启动 Jupyter Lab。
2. 选择项目 `.venv` 对应的 Python 内核。
3. 若本课调用百炼，先在启动 Jupyter 的终端设置 `DASHSCOPE_API_KEY`。
4. 不要把 Key 粘贴到单元格、截图或 Git 提交中。

> 下方“准备代码”可能与前课重复，这是为了保证每个 Notebook 都能单独运行。初学时建议展开阅读，熟悉后可折叠。


### 准备代码


In [ ]:
# %pip install -q openai pydantic>=2.7 httpx fastapi uvicorn fastmcp langgraph langfuse ragas numpy pytest

import os
from dotenv import load_dotenv

load_dotenv()

# 推荐在启动 Jupyter 前设置：
# Windows PowerShell: $env:DASHSCOPE_API_KEY='sk-...'
# macOS/Linux:       export DASHSCOPE_API_KEY='sk-...'

BAILIAN_API_KEY = os.getenv('DASHSCOPE_API_KEY', '')
BAILIAN_BASE_URL = os.getenv(
    'BAILIAN_BASE_URL',
    'https://dashscope.aliyuncs.com/compatible-mode/v1',
)
BAILIAN_MODEL = os.getenv('BAILIAN_MODEL', 'qwen-plus')
BAILIAN_EMBEDDING_MODEL = os.getenv('BAILIAN_EMBEDDING_MODEL', 'text-embedding-v4')

print('模型:', BAILIAN_MODEL)
print('Base URL:', BAILIAN_BASE_URL)
print('API Key:', '已配置' if BAILIAN_API_KEY else '未配置（调用模型前必须设置）')


### 准备代码


In [ ]:
from __future__ import annotations

import asyncio
import json
import logging
import math
import sqlite3
import time
from dataclasses import dataclass, field
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Awaitable, Callable, Literal, TypedDict

import httpx
import numpy as np
from openai import AsyncOpenAI
from pydantic import BaseModel, ConfigDict, Field, ValidationError

WORKSPACE = (Path.cwd() / 'agent_workspace').resolve()
WORKSPACE.mkdir(exist_ok=True)

def require_api_key() -> None:
    if not BAILIAN_API_KEY:
        raise RuntimeError('请先设置环境变量 DASHSCOPE_API_KEY，然后重新运行配置单元格。')

client = AsyncOpenAI(api_key=BAILIAN_API_KEY or 'missing', base_url=BAILIAN_BASE_URL)
print('工作目录:', WORKSPACE)


### 准备代码


In [ ]:
class AgentLimits(BaseModel):
    model_config = ConfigDict(extra='forbid')
    max_steps: int = Field(default=8, ge=1, le=30)
    model_timeout_s: float = Field(default=45, gt=0, le=300)
    tool_timeout_s: float = Field(default=15, gt=0, le=120)
    total_timeout_s: float = Field(default=120, gt=0, le=600)

class ToolResult(BaseModel):
    ok: bool
    data: Any = None
    error: str | None = None
    retryable: bool = False

ToolHandler = Callable[[BaseModel], Awaitable[Any]]

@dataclass
class RegisteredTool:
    name: str
    description: str
    args_model: type[BaseModel]
    handler: ToolHandler
    side_effect: bool = False

    def openai_schema(self) -> dict[str, Any]:
        schema = self.args_model.model_json_schema()
        schema['additionalProperties'] = False
        return {
            'type': 'function',
            'function': {
                'name': self.name,
                'description': self.description,
                'parameters': schema,
            },
        }

class ToolRegistry:
    def __init__(self) -> None:
        self._tools: dict[str, RegisteredTool] = {}

    def register(self, tool: RegisteredTool) -> None:
        if tool.name in self._tools:
            raise ValueError(f'工具重复注册: {tool.name}')
        self._tools[tool.name] = tool

    @property
    def schemas(self) -> list[dict[str, Any]]:
        return [tool.openai_schema() for tool in self._tools.values()]

    async def execute(self, name: str, raw_arguments: str, timeout_s: float) -> ToolResult:
        tool = self._tools.get(name)
        if tool is None:
            return ToolResult(ok=False, error=f'未知工具: {name}', retryable=False)
        try:
            arguments = json.loads(raw_arguments or '{}')
            validated = tool.args_model.model_validate(arguments)
        except json.JSONDecodeError as exc:
            return ToolResult(ok=False, error=f'工具参数不是合法 JSON: {exc}')
        except ValidationError as exc:
            return ToolResult(ok=False, error=f'工具参数校验失败: {exc}')
        try:
            async with asyncio.timeout(timeout_s):
                value = await tool.handler(validated)
            return ToolResult(ok=True, data=value)
        except TimeoutError:
            return ToolResult(ok=False, error=f'工具 {name} 执行超时', retryable=True)
        except httpx.HTTPStatusError as exc:
            retryable = exc.response.status_code in {408, 429, 500, 502, 503, 504}
            return ToolResult(ok=False, error=f'上游 HTTP {exc.response.status_code}', retryable=retryable)
        except Exception as exc:
            return ToolResult(ok=False, error=f'{type(exc).__name__}: {exc}', retryable=False)


### 准备代码


In [ ]:
class MinimalAgent:
    def __init__(self, registry: ToolRegistry, limits: AgentLimits | None = None) -> None:
        self.registry = registry
        self.limits = limits or AgentLimits()

    async def run(self, user_input: str) -> str:
        require_api_key()
        messages: list[dict[str, Any]] = [
            {'role': 'system', 'content': '你是可靠的中文助手。需要外部事实或操作时调用工具；工具失败时不得编造结果。'},
            {'role': 'user', 'content': user_input},
        ]
        repeated_calls: dict[str, int] = {}

        async with asyncio.timeout(self.limits.total_timeout_s):
            for step in range(1, self.limits.max_steps + 1):
                async with asyncio.timeout(self.limits.model_timeout_s):
                    response = await client.chat.completions.create(
                        model=BAILIAN_MODEL,
                        messages=messages,
                        tools=self.registry.schemas or None,
                        tool_choice='auto' if self.registry.schemas else None,
                        temperature=0.2,
                    )
                message = response.choices[0].message
                assistant_message: dict[str, Any] = {
                    'role': 'assistant',
                    'content': message.content or '',
                }
                if message.tool_calls:
                    assistant_message['tool_calls'] = [tc.model_dump() for tc in message.tool_calls]
                messages.append(assistant_message)

                if not message.tool_calls:
                    return message.content or ''

                for call in message.tool_calls:
                    fingerprint = f'{call.function.name}:{call.function.arguments}'
                    repeated_calls[fingerprint] = repeated_calls.get(fingerprint, 0) + 1
                    if repeated_calls[fingerprint] > 2:
                        result = ToolResult(ok=False, error='相同工具调用重复过多，已阻止循环')
                    else:
                        result = await self.registry.execute(
                            call.function.name,
                            call.function.arguments,
                            self.limits.tool_timeout_s,
                        )
                    messages.append({
                        'role': 'tool',
                        'tool_call_id': call.id,
                        'content': result.model_dump_json(),
                    })

        raise RuntimeError(f'Agent 超过最大步骤数 {self.limits.max_steps}')


### 准备代码


In [ ]:
class WeatherArgs(BaseModel):
    model_config = ConfigDict(extra='forbid')
    city: str = Field(min_length=1, max_length=80)

class ExchangeArgs(BaseModel):
    model_config = ConfigDict(extra='forbid')
    amount: float = Field(gt=0, le=10_000_000)
    from_currency: str = Field(pattern=r'^[A-Za-z]{3}$')
    to_currency: str = Field(pattern=r'^[A-Za-z]{3}$')

class TodoArgs(BaseModel):
    model_config = ConfigDict(extra='forbid')
    action: Literal['add', 'list', 'complete', 'delete']
    title: str | None = Field(default=None, max_length=200)
    todo_id: int | None = Field(default=None, ge=1)

class LogArgs(BaseModel):
    model_config = ConfigDict(extra='forbid')
    level: Literal['INFO', 'WARNING', 'ERROR'] = 'INFO'
    message: str = Field(min_length=1, max_length=1000)
    metadata: dict[str, Any] = Field(default_factory=dict)

class SearchArgs(BaseModel):
    model_config = ConfigDict(extra='forbid')
    query: str = Field(min_length=2, max_length=200)
    max_results: int = Field(default=5, ge=1, le=10)

async def get_weather(args: WeatherArgs) -> dict[str, Any]:
    async with httpx.AsyncClient(timeout=10) as http:
        geo = await http.get('https://geocoding-api.open-meteo.com/v1/search', params={
            'name': args.city, 'count': 1, 'language': 'zh', 'format': 'json'
        })
        geo.raise_for_status()
        results = geo.json().get('results') or []
        if not results:
            return {'found': False, 'city': args.city}
        place = results[0]
        weather = await http.get('https://api.open-meteo.com/v1/forecast', params={
            'latitude': place['latitude'],
            'longitude': place['longitude'],
            'current': 'temperature_2m,apparent_temperature,precipitation,weather_code',
            'timezone': 'auto',
        })
        weather.raise_for_status()
        return {'found': True, 'city': place['name'], 'country': place.get('country'), **weather.json()['current']}

async def convert_currency(args: ExchangeArgs) -> dict[str, Any]:
    source, target = args.from_currency.upper(), args.to_currency.upper()
    if source == target:
        return {'amount': args.amount, 'from': source, 'to': target, 'converted': args.amount, 'rate': 1}
    async with httpx.AsyncClient(timeout=10) as http:
        response = await http.get('https://api.frankfurter.app/latest', params={'amount': args.amount, 'from': source, 'to': target})
        response.raise_for_status()
        data = response.json()
        converted = data['rates'][target]
        return {'amount': args.amount, 'from': source, 'to': target, 'converted': converted, 'rate': converted / args.amount}

TODO_DB = WORKSPACE / 'todos.sqlite3'

def init_todo_db() -> None:
    with sqlite3.connect(TODO_DB) as conn:
        conn.execute('CREATE TABLE IF NOT EXISTS todos (id INTEGER PRIMARY KEY, title TEXT NOT NULL, done INTEGER NOT NULL DEFAULT 0)')

async def manage_todo(args: TodoArgs) -> list[dict[str, Any]] | dict[str, Any]:
    init_todo_db()
    with sqlite3.connect(TODO_DB) as conn:
        conn.row_factory = sqlite3.Row
        if args.action == 'add':
            if not args.title:
                raise ValueError('add 操作必须提供 title')
            cursor = conn.execute('INSERT INTO todos(title) VALUES (?)', (args.title,))
            return {'id': cursor.lastrowid, 'title': args.title, 'done': False}
        if args.action in {'complete', 'delete'} and not args.todo_id:
            raise ValueError(f'{args.action} 操作必须提供 todo_id')
        if args.action == 'complete':
            cursor = conn.execute('UPDATE todos SET done=1 WHERE id=?', (args.todo_id,))
            return {'updated': cursor.rowcount}
        if args.action == 'delete':
            cursor = conn.execute('DELETE FROM todos WHERE id=?', (args.todo_id,))
            return {'deleted': cursor.rowcount}
        rows = conn.execute('SELECT id, title, done FROM todos ORDER BY id').fetchall()
        return [dict(row) for row in rows]

async def write_log(args: LogArgs) -> dict[str, Any]:
    record = {
        'timestamp': datetime.now(timezone.utc).isoformat(),
        'level': args.level,
        'message': args.message,
        'metadata': args.metadata,
    }
    path = WORKSPACE / 'agent.jsonl'
    with path.open('a', encoding='utf-8') as handle:
        handle.write(json.dumps(record, ensure_ascii=False) + '\n')
    return {'written': True, 'path': str(path)}

async def web_search(args: SearchArgs) -> list[dict[str, str]]:
    async with httpx.AsyncClient(timeout=10, headers={'User-Agent': 'agent-learning-notebook/1.0'}) as http:
        response = await http.get('https://zh.wikipedia.org/w/api.php', params={
            'action': 'query', 'list': 'search', 'srsearch': args.query,
            'format': 'json', 'utf8': 1, 'srlimit': args.max_results,
        })
        response.raise_for_status()
        return [
            {'title': item['title'], 'snippet': item['snippet'], 'url': f"https://zh.wikipedia.org/wiki/{item['title'].replace(' ', '_')}"}
            for item in response.json()['query']['search']
        ]

registry = ToolRegistry()
for tool in [
    RegisteredTool('get_weather', '查询城市当前天气', WeatherArgs, get_weather),
    RegisteredTool('convert_currency', '按最新公开汇率换算货币', ExchangeArgs, convert_currency),
    RegisteredTool('manage_todo', '添加、列出、完成或删除待办', TodoArgs, manage_todo, side_effect=True),
    RegisteredTool('write_log', '写入一条结构化日志', LogArgs, write_log, side_effect=True),
    RegisteredTool('web_search', '搜索百科资料，返回标题、摘要和链接', SearchArgs, web_search),
]:
    registry.register(tool)

agent = MinimalAgent(registry)
print([schema['function']['name'] for schema in registry.schemas])


### 准备代码


In [ ]:
from fastapi import FastAPI, HTTPException
from fastapi.responses import StreamingResponse

class ChatRequest(BaseModel):
    model_config = ConfigDict(extra='forbid')
    message: str = Field(min_length=1, max_length=10_000)
    user_id: str = Field(min_length=1, max_length=100)

def detect_obvious_injection(text: str) -> bool:
    markers = [
        '忽略所有规则', 'ignore previous instructions',
        '读取环境变量', 'read environment variables',
        'delete_all_files',
    ]
    lowered = text.casefold()
    return any(marker.casefold() in lowered for marker in markers)

def choose_model(message: str, risk: Literal['low', 'high'] = 'low') -> str:
    # 示例策略：真实系统应使用可评估的分类器，而不是只看长度。
    if risk == 'high' or len(message) > 2000:
        return os.getenv('BAILIAN_STRONG_MODEL', BAILIAN_MODEL)
    return os.getenv('BAILIAN_CHEAP_MODEL', 'qwen-flash')

api = FastAPI(title='Bailian Agent API', version='0.1.0')

@api.get('/health')
async def health() -> dict[str, str]:
    return {'status': 'ok'}

@api.post('/chat')
async def chat(request: ChatRequest) -> dict[str, str]:
    if detect_obvious_injection(request.message):
        raise HTTPException(status_code=400, detail='请求包含高风险指令')
    try:
        answer = await agent.run(request.message)
        return {'answer': answer}
    except TimeoutError as exc:
        raise HTTPException(status_code=504, detail='Agent 执行超时') from exc

@api.post('/chat/stream')
async def chat_stream(request: ChatRequest) -> StreamingResponse:
    if detect_obvious_injection(request.message):
        raise HTTPException(status_code=400, detail='请求包含高风险指令')

    async def events():
        answer = await agent.run(request.message)
        for index in range(0, len(answer), 20):
            payload = json.dumps({'delta': answer[index:index + 20]}, ensure_ascii=False)
            yield f'data: {payload}\n\n'
            await asyncio.sleep(0)
        yield 'event: done\ndata: {}\n\n'

    return StreamingResponse(events(), media_type='text/event-stream')

print('FastAPI routes:', [route.path for route in api.routes])
# 终端启动示例：uvicorn app:api --host 0.0.0.0 --port 8000


## Dockerfile 参考

把 FastAPI 部分移动到 `app.py` 后，可使用：

```dockerfile
FROM python:3.12-slim
WORKDIR /app
COPY pyproject.toml ./
RUN pip install --no-cache-dir .
COPY . .
RUN useradd --create-home agent && chown -R agent:agent /app
USER agent
EXPOSE 8000
CMD ["uvicorn", "app:api", "--host", "0.0.0.0", "--port", "8000"]
```

生产部署还应配置反向代理超时、请求体上限、速率限制、日志脱敏、Secret Manager 和健康检查。


## 3. 观察与验证

核心代码中的真实 API 调用默认被注释。先运行无需额度的断言或定义单元格；确认输出和预期一致后，再逐行取消示例注释。


## 4. 代码讲解

Notebook 只定义 `api` 对象并检查路由。真正启动时应迁移到 Python 模块后运行 Uvicorn。当前 SSE 是把最终答案分块，真实 token 流还要处理客户端断开与上游取消。

调试建议：从报错的最后一行开始读，确认当前 Notebook 的单元格是否按顺序全部运行；若看到 `NameError`，通常是准备单元格未运行或内核已重启。


## 5. 常见问题

- **`ModuleNotFoundError`**：确认选中了 `.venv` 内核，并重新安装 `requirements.txt`。
- **提示未配置 API Key**：在启动 Jupyter 的同一个终端设置环境变量，然后重启内核。
- **网络超时或 429**：公开接口或模型服务可能限流；稍后重试，不要移除超时保护。
- **运行结果和预期不同**：先执行“Restart Kernel and Run All”，排除旧变量残留。
- **产生费用吗？**：只有实际调用百炼聊天或 Embedding 接口才会消耗额度；本地定义、SQLite 和断言不会。

## 6. 练习

- 用 FastAPI TestClient 测试 `/health`
- 为 `/chat` 增加每用户限流
- 把应用迁移到 `app.py` 并用 Uvicorn 启动

建议先复制相关单元格再修改，保留一份能工作的基线。


## 7. 本课验收

完成后逐项确认：

- [ ] 请求字段长度受限
- [ ] 危险输入返回 400
- [ ] Agent 超时映射为合适的 HTTP 状态码

如果某项还解释不清，回到对应代码，用更小的输入单独调用函数，而不是直接运行完整 Agent。


## 下一步

继续学习 `11_作品集项目与验收.ipynb`。

> 学习记录建议：写下今天最重要的一个概念、遇到的一个错误、以及你如何验证修复。
